# Options data cleaning

This notebook applies the cleaning pipeline to the AAPL and SPX options datasets, following the methodology discussed for the thesis (see `data_quality_check.ipynb` for the preceding diagnostic pass, which only inspects the data without modifying it).

**Structure:**
- Part 1 — AAPL options cleaning
- Part 2 — SPX options cleaning (added once Part 1 is validated)

**Part 1 sections (this pass):**
1. Setup and imports
2. Load raw data
3. Basic type fixes (dates, strike price scale)
4. `cfadj` (stock split adjustment) — decision, based on diagnostics in `data_quality_check.ipynb`: no adjustment needed

Further steps (ss_flag filter, bid/ask checks, liquidity filter, minimum maturity filter, moneyness/ITM handling, no-arbitrage checks) will be added next.

## Part 1 — AAPL options cleaning

### 1. Setup and imports

In [125]:
import pandas as pd
import numpy as np
from pathlib import Path

In [126]:
# Confirm working directory so relative paths resolve correctly
# (this notebook is expected to sit at the project root, same folder as data_quality_check.ipynb)
print("Current working directory:", Path.cwd())

RAW_DATA_DIR = Path("data/raw")
AAPL_OPTIONS_PATH = RAW_DATA_DIR / "AAPL_options_prices.csv"

print("Looking for AAPL options file at:", AAPL_OPTIONS_PATH.resolve())
print("File exists:", AAPL_OPTIONS_PATH.exists())

Current working directory: c:\Users\glmar\OneDrive\Bemacs Bocconi\TESI\Thesis\Priced-by-intelligence
Looking for AAPL options file at: C:\Users\glmar\OneDrive\Bemacs Bocconi\TESI\Thesis\Priced-by-intelligence\data\raw\AAPL_options_prices.csv
File exists: True


### 2. Load raw data

In [127]:
aapl_raw = pd.read_csv(AAPL_OPTIONS_PATH)

print("Shape:", aapl_raw.shape)
aapl_raw.head()

Shape: (1994694, 15)


,secid,date,exdate,cp_flag,strike_price,best_bid,best_offer,volume,open_interest,optionid,cfadj,ss_flag,index_flag,issuer,exercise_style
0,101594,2020-08-31,2020-09-04,C,100000,29.40,29.70,150,411,135212375,4,0,0,APPLE INC,A
1,101594,2020-08-31,2020-09-04,C,100630,28.85,29.10,691,81,135212376,4,0,0,APPLE INC,A
2,101594,2020-08-31,2020-09-04,C,101250,28.15,28.45,137,240,135212377,4,0,0,APPLE INC,A
3,101594,2020-08-31,2020-09-04,C,101880,27.50,27.85,75,130,135212378,4,0,0,APPLE INC,A
4,101594,2020-08-31,2020-09-04,C,102500,26.90,27.20,323,734,135212379,4,0,0,APPLE INC,A


In [128]:
aapl_raw.dtypes

secid               int64
date                  str
exdate                str
cp_flag               str
strike_price        int64
best_bid          float64
best_offer        float64
volume              int64
open_interest       int64
optionid            int64
cfadj               int64
ss_flag             int64
index_flag          int64
issuer                str
exercise_style        str
dtype: object

### 3. Basic type fixes

Convert `date` and `exdate` to proper datetime objects, and rescale `strike_price` (OptionMetrics stores it multiplied by 1000).

In [129]:
# Work on a copy so the raw dataframe stays untouched for reference
aapl = aapl_raw.copy()

aapl["date"] = pd.to_datetime(aapl["date"])
aapl["exdate"] = pd.to_datetime(aapl["exdate"])

print(aapl[["date", "exdate"]].dtypes)
print("Trading date range:", aapl["date"].min(), "to", aapl["date"].max())
print("Expiration date range:", aapl["exdate"].min(), "to", aapl["exdate"].max())

date      datetime64[us]
exdate    datetime64[us]
dtype: object
Trading date range: 2020-08-31 00:00:00 to 2025-08-29 00:00:00
Expiration date range: 2020-09-04 00:00:00 to 2026-08-21 00:00:00


In [130]:
# OptionMetrics stores strike_price multiplied by 1000
aapl["strike_price"] = aapl["strike_price"] / 1000

aapl["strike_price"].describe()

count    1.994694e+06
mean     1.608919e+02
std      7.507302e+01
min      5.000000e+00
25%      1.075000e+02
50%      1.500000e+02
75%      2.125000e+02
max      4.000000e+02
Name: strike_price, dtype: float64

### 4. Restrict to call options

Following Hutchinson, Lo and Poggio (1994), this analysis is restricted to call options at this stage. Put options are excluded from the dataset rather than reconciled against calls via put-call parity. This keeps option type unambiguous for both the Black-Scholes benchmark and the machine learning models without requiring `cp_flag` as an additional model feature.

In [131]:
# Restrict the dataset to call options only
n_before = len(aapl)
aapl = aapl[aapl["cp_flag"] == "C"].copy()
n_after = len(aapl)

print(f"Rows before: {n_before}")
print(f"Rows after restricting to calls: {n_after}")
print(f"Rows removed: {n_before - n_after}")

Rows before: 1994694
Rows after restricting to calls: 997347
Rows removed: 997347


### 5. Bid/ask validity checks

This section addresses two distinct issues in `best_bid` and `best_offer`: (i) crossed markets, where `best_bid > best_offer`, which cannot represent a genuine tradable quote and are removed unconditionally; (ii) non-positive bids, which are not technically invalid but indicate illiquid, unreliable quotes. 

In [132]:
# Remove crossed markets: best_bid must not exceed best_offer
n_before = len(aapl)
aapl = aapl[aapl["best_bid"] <= aapl["best_offer"]].copy()
n_after = len(aapl)

print(f"Rows before: {n_before}")
print(f"Rows after removing crossed markets (best_bid > best_offer): {n_after}")
print(f"Rows removed: {n_before - n_after}")

Rows before: 997347
Rows after removing crossed markets (best_bid > best_offer): 997316
Rows removed: 31


In [133]:
# Remove rows with non-positive best_bid (zero or negative quotes are not valid tradable prices)
n_before = len(aapl)
aapl = aapl[aapl["best_bid"] > 0].copy()
n_after = len(aapl)

print(f"Rows before: {n_before}")
print(f"Rows after removing non-positive best_bid: {n_after}")
print(f"Rows removed: {n_before - n_after}")


Rows before: 997316
Rows after removing non-positive best_bid: 913369
Rows removed: 83947


### 6. Liquidity filter (volume and open interest)

This section examines `volume` and `open_interest` as liquidity indicators. Options with no trading activity and no open positions are unlikely to carry a reliable price signal. Rows are removed only where both `volume` and `open_interest` are zero on the same day, indicating no trading activity and no outstanding positions on that contract. Using `volume == 0` alone would be too aggressive, since many actively traded contracts simply see no trades on a given day without being illiquid. Similarly, using `open_interest == 0` alone would risk removing contracts with genuine same-day trading activity, since open interest is typically reported as of the prior day's close and can lag newly active positions.

In [134]:
n_before = len(aapl)
aapl = aapl[~((aapl["volume"] == 0) & (aapl["open_interest"] == 0))].copy()
n_after = len(aapl)

print(f"Rows before: {n_before}")
print(f"Rows after liquidity filter: {n_after}")
print(f"Rows removed: {n_before - n_after}")

Rows before: 913369
Rows after liquidity filter: 847997
Rows removed: 65372


## Part 1.2 — AAPL feature engineering

### 7. Remove useless features

In [135]:
columns_to_drop = ["ss_flag", "cfadj", "index_flag", "issuer", "exercise_style", "cp_flag"]
aapl = aapl.drop(columns=columns_to_drop)

print("Remaining columns:", list(aapl.columns))

Remaining columns: ['secid', 'date', 'exdate', 'strike_price', 'best_bid', 'best_offer', 'volume', 'open_interest', 'optionid']


### 8. Merge stock price (S)

Following Hutchinson, Lo and Poggio (1994), moneyness is constructed as `S/K` rather than using the stock price and strike price as separate inputs. This reflects the degree-one homogeneity property of the option pricing function in `(S, K)`, under the assumption that the return distribution of the underlying is independent of the price level (Merton, 1990, Theorem 8.9). Constructing this feature requires the underlying stock price for each option observation, obtained by merging with `AAPL_security_prices.csv` on `date`.

In [136]:
security_prices = pd.read_csv(RAW_DATA_DIR / "AAPL_security_prices.csv")
security_prices["date"] = pd.to_datetime(security_prices["date"])

n_before = len(aapl)
aapl = aapl.merge(security_prices[["date", "close"]], on="date", how="left")
n_after = len(aapl)

print(f"Rows before merge: {n_before}")
print(f"Rows after merge: {n_after}")
print(f"Rows with missing underlying price after merge: {aapl['close'].isna().sum()}")

Rows before merge: 847997
Rows after merge: 847997
Rows with missing underlying price after merge: 0


### 9. Moneyness (S/K)

Moneyness is constructed as `S/K`, following Hutchinson, Lo and Poggio (1994), reflecting the degree-one homogeneity of the option pricing function in `(S, K)` under the assumption that the underlying's return distribution is independent of its price level (Merton, 1990, Theorem 8.9).

In [137]:
aapl["moneyness (S/K)"] = aapl["close"] / aapl["strike_price"]

aapl["moneyness (S/K)"].describe()

count    847997.000000
mean          1.492963
std           2.314207
min           0.416733
25%           0.869822
50%           1.080556
75%           1.497437
max          51.804000
Name: moneyness (S/K), dtype: float64

### 10. Target variable 

The target variable is constructed as the option mid-price, `(best_bid + best_offer) / 2`, computed after the bid/ask validity checks in Section 5 to ensure both quotes are genuine.

In [138]:
aapl["price"] = (aapl["best_bid"] + aapl["best_offer"]) / 2

aapl["price"].describe()

count    847997.000000
mean         35.033130
std          42.696374
min           0.010000
25%           1.235000
50%          16.750000
75%          57.600000
max         254.550000
Name: price, dtype: float64

In [139]:
aapl.head()

,secid,date,exdate,strike_price,best_bid,best_offer,volume,open_interest,optionid,close,moneyness (S/K),price
0,101594,2020-08-31,2020-09-04,100.00,29.40,29.70,150,411,135212375,129.04,1.290400,29.550
1,101594,2020-08-31,2020-09-04,100.63,28.85,29.10,691,81,135212376,129.04,1.282321,28.975
2,101594,2020-08-31,2020-09-04,101.25,28.15,28.45,137,240,135212377,129.04,1.274469,28.300
3,101594,2020-08-31,2020-09-04,101.88,27.50,27.85,75,130,135212378,129.04,1.266588,27.675
4,101594,2020-08-31,2020-09-04,102.50,26.90,27.20,323,734,135212379,129.04,1.258927,27.050


### 11. Merge historical volatility (60-day window)

Following Hutchinson, Lo and Poggio (1994), sigma is estimated as the realized volatility over a fixed 60-day trailing window, applied uniformly across all options regardless of their individual time to maturity. This choice may be revisited in favor of a per-option maturity-matched window later in the analysis.

In [140]:
historical_volatility = pd.read_csv(RAW_DATA_DIR / "AAPL_historical_volatility.csv")
historical_volatility["date"] = pd.to_datetime(historical_volatility["date"])

hv_60 = historical_volatility[historical_volatility["days"] == 60]

In [141]:
n_before = len(aapl)
aapl = aapl.merge(hv_60[["date", "volatility"]], on="date", how="left")
n_after = len(aapl)

print(f"Rows before merge: {n_before}")
print(f"Rows after merge: {n_after}")
print(f"Rows with missing underlying price after merge: {aapl['close'].isna().sum()}")
aapl.head()

Rows before merge: 847997
Rows after merge: 847997
Rows with missing underlying price after merge: 0


,secid,date,exdate,strike_price,best_bid,best_offer,volume,open_interest,optionid,close,moneyness (S/K),price,volatility
0,101594,2020-08-31,2020-09-04,100.00,29.40,29.70,150,411,135212375,129.04,1.290400,29.550,0.375779
1,101594,2020-08-31,2020-09-04,100.63,28.85,29.10,691,81,135212376,129.04,1.282321,28.975,0.375779
2,101594,2020-08-31,2020-09-04,101.25,28.15,28.45,137,240,135212377,129.04,1.274469,28.300,0.375779
3,101594,2020-08-31,2020-09-04,101.88,27.50,27.85,75,130,135212378,129.04,1.266588,27.675,0.375779
4,101594,2020-08-31,2020-09-04,102.50,26.90,27.20,323,734,135212379,129.04,1.258927,27.050,0.375779


### 12. Merge risk-free rate (90-day tenor)

For each date, the zero-coupon rate closest to a 90-day maturity is selected from the Zero_Curve file, rather than requiring an exact match, since available tenors vary slightly around 90 days (89, 90, 91, ...) rather than following fixed standardized bins.

In [142]:
riskfree = pd.read_csv(RAW_DATA_DIR / "riskfree_rate.csv")
riskfree["date"] = pd.to_datetime(riskfree["date"])

In [143]:
riskfree["days_diff"] = (riskfree["days"] - 90).abs()
riskfree_90 = riskfree.loc[riskfree.groupby("date")["days_diff"].idxmin()][["date", "rate"]]

n_before = len(aapl)
aapl = aapl.merge(riskfree_90, on="date", how="left")
n_after = len(aapl)

print(f"Rows before merge: {n_before}")
print(f"Rows after merge: {n_after}")
print(f"Rows with missing rate after merge: {aapl['rate'].isna().sum()}")

aapl.head()

Rows before merge: 847997
Rows after merge: 847997
Rows with missing rate after merge: 0


,secid,date,exdate,strike_price,best_bid,best_offer,volume,open_interest,optionid,close,moneyness (S/K),price,volatility,rate
0,101594,2020-08-31,2020-09-04,100.00,29.40,29.70,150,411,135212375,129.04,1.290400,29.550,0.375779,0.214845
1,101594,2020-08-31,2020-09-04,100.63,28.85,29.10,691,81,135212376,129.04,1.282321,28.975,0.375779,0.214845
2,101594,2020-08-31,2020-09-04,101.25,28.15,28.45,137,240,135212377,129.04,1.274469,28.300,0.375779,0.214845
3,101594,2020-08-31,2020-09-04,101.88,27.50,27.85,75,130,135212378,129.04,1.266588,27.675,0.375779,0.214845
4,101594,2020-08-31,2020-09-04,102.50,26.90,27.20,323,734,135212379,129.04,1.258927,27.050,0.375779,0.214845


### 13. Dividend yield (q)

#### Exclude same-day expirations (T = 0)

Options with zero days to expiration are excluded, since T = 0 causes a division by zero in the Black-Scholes d1/d2 terms. This exclusion is applied regardless of the broader minimum-maturity threshold decision (deferred to post-training analysis, following Hutchinson, Lo and Poggio, 1994), since it addresses a mathematical requirement of the pricing formula itself rather than a data-quality judgment.

In [147]:
aapl["days_to_maturity"] = (aapl["exdate"] - aapl["date"]).dt.days
aapl["T"] = aapl["days_to_maturity"] / 365

aapl["T"].describe()

count    847997.000000
mean          0.305302
std           0.279779
min           0.000000
25%           0.065753
50%           0.213699
75%           0.504110
max           1.000000
Name: T, dtype: float64

In [148]:
n_before = len(aapl)
aapl = aapl[aapl["T"] > 0].copy()
n_after = len(aapl)

print(f"Rows before: {n_before}")
print(f"Rows after excluding T = 0: {n_after}")
print(f"Rows removed: {n_before - n_after}")

Rows before: 847997
Rows after excluding T = 0: 838276
Rows removed: 9721


Following the OptionMetrics IvyDB methodology, `q` is computed as the most recently declared regular dividend divided by the underlying's closing price, held constant until the next declared dividend. Only actual declared dividends (`distr_type == 1`) are used, excluding cancelled distributions and yield projections (`distr_type == '%'`, where the `amount` field represents a yield rather than a dollar amount).